In [2]:
import os
from openai import OpenAI, pydantic_function_tool
import rich
import requests
import json
from pydantic import BaseModel, Field

In [3]:
API_KEY = os.environ.get('AIHUBMIX_API_KEY')
BASE_URL = os.environ.get('AIHUBMIX_BASE_URL')
MODEL = "gpt-5-nano"

openai = OpenAI(api_key=API_KEY, base_url=BASE_URL)

**Calling the same function multiple times when response from Chat API and Responses API requires to call function multiple times**

The Pydantic-generated function structure is acceptable in OpenAI's Chat API, but the Responses API requires a slightly different structure.

In [4]:
class GetWeather(BaseModel):
    latitude: float = Field(..., description="Latitude of the location")
    longitude: float = Field(..., description="Longitude of the location")

def get_weather(latitude, longitude):
    response = requests.get(f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m,wind_speed_10m&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m")
    data = response.json()
    print(f"get_weather function called to get weather for latitude = {latitude}, longitude = {longitude}")
    print(f"And result is  = {data['current']['temperature_2m']}")
    return data['current']['temperature_2m']

# Notice the function property in the output, which is not acceptable in Responses API
rich.print(pydantic_function_tool(GetWeather))

{
    'type': 'function',
    'function': {
        'name': 'GetWeather',
        'strict': True,
        'parameters': {
            'properties': {
                'latitude': {'description': 'Latitude of the location', 'title': 'Latitude', 'type': 'number'},
                'longitude': {'description': 'Longitude of the location', 'title': 'Longitude', 'type': 'number'}
            },
            'required': ['latitude', 'longitude'],
            'title': 'GetWeather',
            'type': 'object',
            'additionalProperties': False
        }
    }
}

# Chat Completion API

https://platform.openai.com/docs/guides/function-calling?api-mode=chat

First Step where model will responed with tool call request

In [5]:
messages=[
    {"role": "developer", "content": "你是一个有用的私人助手，能够提供城市实时天气的服务。回复必须专业、严谨。"},
    # {"role": "user", "content": "What's the weather like in Karachi and Lahore?"}
    # {"role": "user", "content": "NYC"}
    {"role": "user", "content": "Berlin and Paris"} # This time we are sending multiple cities
]
tools = [pydantic_function_tool(GetWeather)]
response = openai.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools = tools
)

rich.print(response.choices[0])
print("Finish Reason = ", response.choices[0].finish_reason)
# Chat API returns tool_calls for all expected tools in single response
rich.print(response.choices[0].message.tool_calls)
print("Number of tool calls: ",len(response.choices[0].message.tool_calls))

Choice(
    finish_reason='tool_calls',
    index=0,
    logprobs=None,
    message=ChatCompletionMessage(
        content=None,
        refusal=None,
        role='assistant',
        annotations=None,
        audio=None,
        function_call=None,
        tool_calls=[
            ChatCompletionMessageFunctionToolCall(
                id='call_gdN7s2LoFPhVjuT5DF5i9ddJ',
                function=Function(arguments='{"latitude": 52.52, "longitude": 13.405}', name='GetWeather'),
                type='function',
                custom={}
            ),
            ChatCompletionMessageFunctionToolCall(
                id='call_sCr1e5XRwQ2FG3Q1XliPdNUK',
                function=Function(arguments='{"latitude": 48.8566, "longitude": 2.3522}', name='GetWeather'),
                type='function',
                custom={}
            )
        ]
    )
)

Finish Reason =  tool_calls


[
    ChatCompletionMessageFunctionToolCall(
        id='call_gdN7s2LoFPhVjuT5DF5i9ddJ',
        function=Function(arguments='{"latitude": 52.52, "longitude": 13.405}', name='GetWeather'),
        type='function',
        custom={}
    ),
    ChatCompletionMessageFunctionToolCall(
        id='call_sCr1e5XRwQ2FG3Q1XliPdNUK',
        function=Function(arguments='{"latitude": 48.8566, "longitude": 2.3522}', name='GetWeather'),
        type='function',
        custom={}
    )
]

Number of tool calls:  2


Second Step where we are calling the `get_weather` function and sending the response back to Chat API.

But this time we are running a loop on `tool_call` to call all the functions

In [6]:
# Defining a function in which we will loop through the tool_calls,
# invoke the corresponding functions, and update the response.
# Plus, we will append a new message for each function call to promptMessages.
def handle_tool_call(promptMessages, responseMessage):
    # Looping on tool_calls
    for tool_call in responseMessage.tool_calls:
        arguments = json.loads(tool_call.function.arguments)
        weather = get_weather(**arguments)
        new_message = {
            "role": "tool",
            "content": str(weather),
            "tool_call_id": tool_call.id
        }
        promptMessages.append(new_message)

In [7]:
if response.choices[0].finish_reason == "tool_calls": # Check if finish_reason is tool_calls
    # Important: first we will append the previous message (response.choices[0].message) in
    # message history and then iterate through tool_calls
    messages.append(response.choices[0].message)
    handle_tool_call(messages, response.choices[0].message)
    response2 = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    print("Model Response2 = ",response2.choices[0].message.content)
    print("Finish Reason = ",response2.choices[0].finish_reason)

get_weather function called to get weather for latitude = 52.52, longitude = 13.405
And result is  = 4.0
get_weather function called to get weather for latitude = 48.8566, longitude = 2.3522
And result is  = 8.2
Model Response2 =  以下是您请求的两座城市实时气温（单位：℃）：

- 柏林：4.0°C
- 巴黎：8.2°C

如需，我可以继续提供风速、湿度、降水概率、风向、空气质量等更多实时信息，或给出未来几小时的温度趋势。需要我进一步查询吗？
Finish Reason =  stop


In [9]:
rich.print(messages)

[
    {'role': 'developer', 'content': '你是一个有用的私人助手，能够提供城市实时天气的服务。回复必须专业、严谨。'},
    {'role': 'user', 'content': 'Berlin and Paris'},
    ChatCompletionMessage(
        content=None,
        refusal=None,
        role='assistant',
        annotations=None,
        audio=None,
        function_call=None,
        tool_calls=[
            ChatCompletionMessageFunctionToolCall(
                id='call_gdN7s2LoFPhVjuT5DF5i9ddJ',
                function=Function(arguments='{"latitude": 52.52, "longitude": 13.405}', name='GetWeather'),
                type='function',
                custom={}
            ),
            ChatCompletionMessageFunctionToolCall(
                id='call_sCr1e5XRwQ2FG3Q1XliPdNUK',
                function=Function(arguments='{"latitude": 48.8566, "longitude": 2.3522}', name='GetWeather'),
                type='function',
                custom={}
            )
        ]
    ),
    {'role': 'tool', 'content': '4.0', 'tool_call_id': 'call_gdN7s2LoFPhVjuT5DF5i9ddJ'},
    {'role': 'tool', 'content': '8.2', 'tool_call_id': 'call_sCr1e5XRwQ2FG3Q1XliPdNUK'}
]

# Responses API

https://platform.openai.com/docs/guides/function-calling?api-mode=responses

In [10]:
rich.print(pydantic_function_tool(GetWeather))

{
    'type': 'function',
    'function': {
        'name': 'GetWeather',
        'strict': True,
        'parameters': {
            'properties': {
                'latitude': {'description': 'Latitude of the location', 'title': 'Latitude', 'type': 'number'},
                'longitude': {'description': 'Longitude of the location', 'title': 'Longitude', 'type': 'number'}
            },
            'required': ['latitude', 'longitude'],
            'title': 'GetWeather',
            'type': 'object',
            'additionalProperties': False
        }
    }
}

To use pydantic-generated function, we need to use `openai.responses.parse()` function call

https://github.com/openai/openai-python/blob/main/examples/responses/structured_outputs_tools.py

Sometimes Responses API is not sending weather function call for all cities in one go.
It is totally depends on model how it will organized the call of function, all in one go or one by one

In [11]:
messages=[
    {"role": "developer", "content": "You are a helpful assistant and provide update on weather in a city. Response should be courteous and professional."},
    # {"role": "user", "content": "What's the weather like in Karachi and Lahore?"}
    {"role": "user", "content": "Berlin and Paris"}
]
# messages=[
#     {"role": "developer", "content": "You are a helpful assistant and provide update on weather in all the cities as asked by user. Response should be courteous and professional."},
#     {"role": "user", "content": "What's the weather in following cities Karachi and Lahore?"}
# ]
tools = [pydantic_function_tool(GetWeather)]

# Sometimes Responses API is not sending weather function call for all cities in one go.
# It is totally depends on model how it will organized the call of function, all in one go or one by one
response = openai.responses.parse(
    model=MODEL,
    input=messages,
    tools = tools
)

print("Status = ",response.status) # Status will not indicate the tool call
print(response.output_text) # Empty
rich.print(response.output)
# In this case Responses API calling tools one by one. We need to send the response for the first
# tool call to the API, after which the Responses API will send the next tool call.

print("Number of tool calls: ",len(response.output))
# rich.print(response)

Status =  completed



[
    ResponseReasoningItem(
        id='rs_09a515ebd18cc38f0069a65f1a07d88195be36d210c6899bed',
        summary=[],
        type='reasoning',
        content=None,
        encrypted_content=None,
        status=None
    ),
    ParsedResponseFunctionToolCall(
        arguments='{"latitude":52.52,"longitude":13.405}',
        call_id='call_MslkHd3PKrEBNPrxvc17zyuz',
        name='GetWeather',
        type='function_call',
        id='fc_09a515ebd18cc38f0069a65f1cf2048195a38b410820788e6d',
        status='completed',
        parsed_arguments=GetWeather(latitude=52.52, longitude=13.405)
    ),
    ParsedResponseFunctionToolCall(
        arguments='{"latitude":48.8566,"longitude":2.3522}',
        call_id='call_rr5lYRMnl4NbrQs9uRG1LcPO',
        name='GetWeather',
        type='function_call',
        id='fc_09a515ebd18cc38f0069a65f1cf2188195b5efcaf65b27ca8a',
        status='completed',
        parsed_arguments=GetWeather(latitude=48.8566, longitude=2.3522)
    )
]

Number of tool calls:  3


In case of weather function it was random for me so we are sing example of different function to test multiple calls of same function in single response

In [12]:
class SendEmail(BaseModel):
    to: str = Field(..., description="Email address of the recipient")
    subject: str = Field(..., description="Subject of the email")
    body: str = Field(..., description="Body of the email")


def send_email(to, subject, body):
    print(f"Tool send_email Sending email to {to} with subject {subject}")
    print(f"Body: {body}")
    print(f"Email Tool call completed")
    return "Email Sent"

### Using old way of sending history messages in every call

First Step where model will responed with tool call request

In [22]:
messages=[
    {"role": "developer", "content": "You are a helpful assistant, you can send email about weather in a city. Email should be courteous and professional."},
    {"role": "user", "content": "send an email to first@gmail.com and second@gmail.com saying Hello"}
]
tools = [pydantic_function_tool(SendEmail)]

response = openai.responses.parse(
    model=MODEL,
    input=messages,
    tools = tools
)

print("Status = ",response.status) # Status will not indicate the tool call
print(response.output_text) # Empty
rich.print(response.output)

print("Number of tool calls: ",len(response.output))
# rich.print(response)

Status =  completed



[
    ResponseReasoningItem(
        id='rs_08fa6f02c87043030069a6835d43ac81908643b408e1e719d7',
        summary=[],
        type='reasoning',
        content=None,
        encrypted_content=None,
        status=None
    ),
    ParsedResponseFunctionToolCall(
        arguments='{"to":"first@gmail.com","subject":"Hello","body":"Hello"}',
        call_id='call_rybt8liT5ATJ7HYHIyGgvrrD',
        name='SendEmail',
        type='function_call',
        id='fc_08fa6f02c87043030069a683628f2881909973dcd0f76ba544',
        status='completed',
        parsed_arguments=SendEmail(to='first@gmail.com', subject='Hello', body='Hello')
    ),
    ParsedResponseFunctionToolCall(
        arguments='{"to":"second@gmail.com","subject":"Hello","body":"Hello"}',
        call_id='call_6Q2IhydoX0Aiz6Bc46dagoVo',
        name='SendEmail',
        type='function_call',
        id='fc_08fa6f02c87043030069a683628f448190aa50b14c3ff4bb74',
        status='completed',
        parsed_arguments=SendEmail(to='second@gmail.com', subject='Hello', body='Hello')
    )
]

Number of tool calls:  3


In [23]:
rich.print(response)

ParsedResponse[NoneType](
    id='resp_08fa6f02c87043030069a6835c941c81909ca7df3fad55bb69',
    created_at=1772520284.0,
    error=None,
    incomplete_details=None,
    instructions=None,
    metadata={},
    model='gpt-5-nano',
    object='response',
    output=[
        ResponseReasoningItem(
            id='rs_08fa6f02c87043030069a6835d43ac81908643b408e1e719d7',
            summary=[],
            type='reasoning',
            content=None,
            encrypted_content=None,
            status=None
        ),
        ParsedResponseFunctionToolCall(
            arguments='{"to":"first@gmail.com","subject":"Hello","body":"Hello"}',
            call_id='call_rybt8liT5ATJ7HYHIyGgvrrD',
            name='SendEmail',
            type='function_call',
            id='fc_08fa6f02c87043030069a683628f2881909973dcd0f76ba544',
            status='completed',
            parsed_arguments=SendEmail(to='first@gmail.com', subject='Hello', body='Hello')
        ),
        ParsedResponseFunctionToolCall(
            arguments='{"to":"second@gmail.com","subject":"Hello","body":"Hello"}',
            call_id='call_6Q2IhydoX0Aiz6Bc46dagoVo',
            name='SendEmail',
            type='function_call',
            id='fc_08fa6f02c87043030069a683628f448190aa50b14c3ff4bb74',
            status='completed',
            parsed_arguments=SendEmail(to='second@gmail.com', subject='Hello', body='Hello')
        )
    ],
    parallel_tool_calls=True,
    temperature=1.0,
    tool_choice='auto',
    tools=[
        FunctionTool(
            name='SendEmail',
            parameters={
                'properties': {
                    'to': {'description': 'Email address of the recipient', 'title': 'To', 'type': 'string'},
                    'subject': {'description': 'Subject of the email', 'title': 'Subject', 'type': 'string'},
                    'body': {'description': 'Body of the email', 'title': 'Body', 'type': 'string'}
                },
                'required': ['to', 'subject', 'body'],
                'title': 'SendEmail',
                'type': 'object',
                'additionalProperties': False
            },
            strict=True,
            type='function',
            description=None
        )
    ],
    top_p=1.0,
    background=False,
    conversation=None,
    max_output_tokens=None,
    max_tool_calls=None,
    previous_response_id=None,
    prompt=None,
    prompt_cache_key=None,
    prompt_cache_retention=None,
    reasoning=Reasoning(effort='medium', generate_summary=None, summary=None),
    safety_identifier=None,
    service_tier='default',
    status='completed',
    text=ResponseTextConfig(format=ResponseFormatText(type='text'), verbosity='medium'),
    top_logprobs=0,
    truncation='disabled',
    usage=ResponseUsage(
        input_tokens=116,
        input_tokens_details=InputTokensDetails(cached_tokens=0),
        output_tokens=808,
        output_tokens_details=OutputTokensDetails(reasoning_tokens=704),
        total_tokens=924
    ),
    user=None,
    completed_at=1772520291,
    content_filters=[
        {
            'blocked': False,
            'source_type': 'completion',
            'content_filter_raw': [],
            'content_filter_results': {},
            'content_filter_offsets': {'start_offset': 0, 'end_offset': 4762, 'check_offset': 0}
        }
    ],
    frequency_penalty=0.0,
    presence_penalty=0.0,
    store=True
)

Second Step where we are calling the `send_email` function and sending the response back to Responses API.

But this time we are running a loop on `response.output` to call all the functions

In [16]:
# Defining a function in which We will call the relevant function
# and return the new message object
def handle_tool_call_responses(tool_call):
    result = send_email(tool_call.parsed_arguments.to, tool_call.parsed_arguments.subject,tool_call.parsed_arguments.body)

    new_message = {
        "type": "function_call_output",
        "call_id": tool_call.call_id,
        "output": str(result)
    }
    return new_message

In [25]:
# Logic here is bit different then what we have done in Chat API
tool_call_results = []
for tool_call in response.output:
    if tool_call.type == "function_call": # check if the output's type is function_call
        # Adding the result of tool calls individually
        tool_call_results.append(handle_tool_call_responses(tool_call))

Tool send_email Sending email to first@gmail.com with subject Hello
Body: Hello
Email Tool call completed
Tool send_email Sending email to second@gmail.com with subject Hello
Body: Hello
Email Tool call completed


In [29]:
# deleting the parsed_arguments from the response.output because to
# sending messages as history to the API, we need to remove the parsed_arguments
for item in response.output:
    if(hasattr(item, "parsed_arguments")): 
        del item.parsed_arguments

In [30]:
# For multiple tool calls, we need to append the output of each tool call individually.
# This is different from Chat API in which we can just send message object which have all the tool calls
# extend function it adds each item from the iterable one by one.
messages.extend(response.output)
messages.extend(tool_call_results) # Adding the result of tool calls individually
# rich.print(messages)
response2 = openai.responses.parse(model=MODEL, input=messages,tools = tools)
print("Model Response2 output_text = ",response2.output_text)
# rich.print("Model Response2 = ",response2)
print("Status = ",response2.status)

Model Response2 output_text =  Both emails have been sent.

- To: first@gmail.com
  - Subject: Hello
  - Body: Hello

- To: second@gmail.com
  - Subject: Hello
  - Body: Hello

If you’d like a different subject or additional message, I can update and resend.
Status =  completed


### Using new way of conversation state by sending perivous reponse id

First Step where model will responed with tool call request

In [31]:
# This section is same as above
messages=[
    {"role": "developer", "content": "You are a helpful assistant, you can send email about weather in a city. Email should be courteous and professional."},
    {"role": "user", "content": "send an email to first@gmail.com and second@gmail.com saying Hello"}
]
tools = [pydantic_function_tool(SendEmail)]

response = openai.responses.parse(
    model=MODEL,
    input=messages,
    tools = tools
)

print("Status = ",response.status) # Status will not indicate the tool call
print(response.output_text) # Empty
rich.print(response.output)

print("Number of tool calls: ",len(response.output))
# rich.print(response)

Status =  completed



[
    ResponseReasoningItem(
        id='rs_0510640bebbf8bdb0069a685c3512c8197a7fc9c97b3db5d41',
        summary=[],
        type='reasoning',
        content=None,
        encrypted_content=None,
        status=None
    ),
    ParsedResponseFunctionToolCall(
        arguments='{"to":"first@gmail.com","subject":"Hello","body":"Hello. Best regards."}',
        call_id='call_IGpYJiBoR8sZI4AQQz8llTl5',
        name='SendEmail',
        type='function_call',
        id='fc_0510640bebbf8bdb0069a685cc5408819797d1cbfd5fa768e3',
        status='completed',
        parsed_arguments=SendEmail(to='first@gmail.com', subject='Hello', body='Hello. Best regards.')
    ),
    ParsedResponseFunctionToolCall(
        arguments='{"to":"second@gmail.com","subject":"Hello","body":"Hello. Best regards."}',
        call_id='call_BipMmYV4vKNsgSaAvySiE6BZ',
        name='SendEmail',
        type='function_call',
        id='fc_0510640bebbf8bdb0069a685cc542481978a3215eb319eb49d',
        status='completed',
        parsed_arguments=SendEmail(to='second@gmail.com', subject='Hello', body='Hello. Best regards.')
    )
]

Number of tool calls:  3


Second Step where we are calling the `send_email` function and sending the response back to Responses API.

But this time we are running a loop on `response.output` to call all the functions

In [32]:
# This section is same as above

# Call the relevant function and return the output
def handle_tool_call_responses(tool_call):
    result = send_email(tool_call.parsed_arguments.to, tool_call.parsed_arguments.subject,tool_call.parsed_arguments.body)

    new_message = {
        "type": "function_call_output",
        "call_id": tool_call.call_id,
        "output": str(result)
    }
    return new_message

The only difference in below section is how messages are sent.

In [ ]:
# The only difference in this section is how messages are sent.

# Logic here is bit different then what we have done in Chat API
tool_call_results = []
for tool_call in response.output:
    if tool_call.type == "function_call": # check if the output's type is function_call
        tool_call_results.append(handle_tool_call_responses(tool_call))

# Not needed now because we are using response.id
# for item in response.output:
#     del item.parsed_arguments

# Not needed now because we are using response.id
# messages.extend(response.output)

# Emptying the messages array because we are sending the previous response id,
# therefore we don't need to send the previous message
messages = []
# Adding the result of tool calls individually
messages.extend(tool_call_results)
# rich.print(messages)
response2 = openai.responses.parse(model=MODEL, input=messages,tools = tools, previous_response_id=response.id)
print("Model Response2 output_text = ",response2.output_text)
# rich.print("Model Response2 = ",response2)
print("Status = ",response2.status)

Tool send_email Sending email to first@gmail.com with subject Hello
Body: Hello!
Email Tool call completed
Tool send_email Sending email to second@gmail.com with subject Hello
Body: Hello!
Email Tool call completed
Model Response2 output_text =  I have sent an email saying "Hello!" to both first@gmail.com and second@gmail.com. If you need further assistance, feel free to ask!
Status =  completed
